<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Mini-Project: Classification Case Study

*Session 5 · Notebook 03.11 · Mini-Project · Student version*

## About this notebook

This is a hands-on **mini-project** for the classification module. It takes the maternal-health case study end to end: you first prepare the data with the full preprocessing workflow (the same Tasks 1 to 5 as the preprocessing case study), then go one step further and **build and compare classification models** - Logistic Regression, a Decision Tree and a Random Forest - to predict gestational diabetes. The goal is to practise the whole path from a raw file to an evidence-based model recommendation.

## About the exercises

Each task appears as two cells: a **question** (markdown, sometimes with a `> *Hint:*`) and an empty **`# Your turn`** cell for you to write your answer. Work top to bottom, since later tasks reuse variables you create earlier; your coach has the worked solutions.

## The case study

You have been presented with a dataset containing visceral adipose tissue measurements taken during pregnancy. This dataset ([available on PhysioNet](https://physionet.org/content/maternal-visceral-adipose/1.0.0/)) comes from a cohort study of pregnant women up to 20 weeks of pregnancy, followed until delivery. Measurement of maternal visceral adipose tissue (VAT) was performed during a routine obstetric ultrasound. At the same time, a biometric evaluation was carried out and information was obtained from prenatal care. Gestational outcomes, including gestational diabetes mellitus (GDM), were obtained by evaluating the patients' medical records at the hospitals where the births took place. The data was collected as part of a study that sought to evaluate whether maternal VAT could predict GDM at the time of delivery.

The variables in the dataset include maternal age, previous diabetes, blood pressure (on the same day as the VAT measurement), VAT (in the periumbilical region), gestational age at inclusion, number of pregnancies, first fasting glucose level and pre-gestational body mass index (BMI). It also records the pregnancy outcomes: gestational age at birth, type of delivery (vaginal or caesarean section), child birth weight, and the diagnosis of GDM.

The study sample consisted of a cohort of 154 women recruited between October 2016 and December 2017 at the Ultrasound Department of the Murialdo Teaching Health Center in Porto Alegre, Brazil. Participants were followed until delivery. Of the 154 women selected initially, 21 (13%) were lost to follow-up, giving a final sample of 133 women. The inclusion criteria were a singleton pregnancy and a gestational age of 20 weeks or less. The exclusion criterion was pre-existing type 1 or type 2 diabetes mellitus.

## Data description

The intake data (first 20 weeks of pregnancy) and the delivery outcomes (gestational diabetes, type of delivery, newborn weight and gestational age) are recorded in `visceral_fat.csv`. The variables are:

1. **number:** unique ID for the case.
2. **age (years):** age in years.
3. **ethnicity:** ethnicity (0 = white, 1 = not white).
4. **diabetes mellitus:** previous diabetes mellitus (0 = no, 1 = yes).
5. **mean diastolic bp (mmhg):** mean diastolic blood pressure in mmHg.
6. **mean systolic bp (mmhg):** mean systolic blood pressure in mmHg.
7. **central armellini fat (mm):** maternal visceral adipose tissue measurement in mm.
8. **current gestational age:** gestational age at inclusion (weeks, days of pregnancy).
9. **pregnancies (number):** number of pregnancies.
10. **first fasting glucose (mg/dl):** first measured fasting glucose.
11. **bmi pregestational (kg/m):** pre-gestational body mass index.
12. **gestational age at birth:** gestational age at birth (weeks, days of pregnancy).
13. **type of delivery:** 0 = vaginal birth, 1 = caesarean section.
14. **child birth weight (g):** birth weight in grams.
15. **gestational dm:** current gestational diabetes (0 = no, 1 = yes). This is the target.

Missing data is indicated by an empty cell.

## Your task

Your task is to **prepare the data and then build and compare classification models**. The steps are:

1. Perform an initial data assessment and check what kind of issues the data presents.
2. Handle the missing data: evaluate and analyse the impact of the different strategies for missing values.
3. Consider whether the data needs enrichment by creating calculated fields (for example, pulse pressure).
4. Given that the data is highly imbalanced, propose and apply ways to balance it.
5. Apply normalization/standardization to the data.
6. Train and evaluate **classification models** (Logistic Regression, Decision Tree, Random Forest) and compare them to recommend the best predictor of gestational diabetes.

The sections below work through each of these tasks in turn.

## Index

0. [Setup](#setup)
1. [Task 1: Initial data assessment](#task1)
2. [Task 2: Handling missing data](#task2)
3. [Task 3: Data enrichment and feature preparation](#task3)
4. [Task 4: Handling the imbalanced target](#task4)
5. [Task 5: Normalization and standardization](#task5)
6. [Task 6: Apply and evaluate classification models](#task6)
7. [Wrap-up](#wrap)

<a id="setup"></a>
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score)

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")

<a id="task1"></a>
## Task 1: Initial data assessment

Before changing anything, understand what you have. Never clean blind.

### Exercise 1: Load and take a first look

Read `../../datasets/visceral_fat.csv` into a DataFrame called `df`. Show its shape, `df.head()`, and `df.info()`. Note which columns are numeric and which came in as `object` (text).

In [ ]:
# Your turn


### Exercise 2: Find the problems

A good assessment lists concrete issues. Report: (a) missing values per column, (b) the value counts of the target `gestational dm`, and (c) the value counts of `diabetes mellitus` (prior diabetes). What stands out?

In [ ]:
# Your turn


### Exercise 3: Spot the columns stored as text

Look at the first few values of `current gestational age` and `gestational age at birth`, and their dtypes. Why can't a model use them as they are?

> *Hint:* print `df[['current gestational age', 'gestational age at birth']].head()`.

In [ ]:
# Your turn


<a id="task2"></a>
## Task 2: Handling missing data

Some models cannot run with gaps in the data, and filling them carelessly can bias the result. So quantify the missingness first, then choose a strategy.

### Exercise 4: Quantify the missingness

Print, for each column with gaps, the number and percentage of missing values. Which single column is the real problem?

In [ ]:
# Your turn


### Exercise 5: Evaluate the strategies, then demonstrate one

In a comment, weigh the options for `first fasting glucose` (22% missing): dropping the column, dropping the rows, or imputing. Then demonstrate **median imputation** on that column (on a copy of `df`) and confirm the gaps are filled.

> *Note:* this shows the effect. In Task 5 we impute *inside a pipeline* so the fill values are learned from the training data only (no leakage).

In [ ]:
# Your turn


<a id="task3"></a>
## Task 3: Data enrichment and feature preparation

Create useful new features from the raw columns, make the text columns usable, and drop the columns that should not be model inputs.

### Exercise 6: Parse gestational age into numeric weeks

Write a function that turns a `'weeks,days'` string into a number of weeks (days as a fraction of 7), and use it to create `current_ga_weeks` from `current gestational age`. Check the result.

> *Hint:* `'12,1'` should become `12 + 1/7 = 12.14`. Split the string on the comma.

In [ ]:
# Your turn


### Exercise 7: Create a calculated field (pulse pressure)

The brief suggests enriching the data with calculated fields such as **pulse pressure**, the gap between systolic and diastolic blood pressure. Create a `pulse_pressure` column.

> *Hint:* pulse pressure = systolic minus diastolic.

In [ ]:
# Your turn


### Exercise 8: Drop the columns that should not be inputs, then build X and y

Some columns should not be model features:

- `number` is just an ID.
- `diabetes mellitus` is near-constant (from Task 1).
- `current gestational age` is now replaced by `current_ga_weeks`.
- `gestational age at birth`, `type of delivery`, `child birth weight (g)` are **delivery outcomes**, known only *after* the pregnancy, so using them to predict GDM would be target leakage.

Drop those, set `y = df['gestational dm']`, and put the remaining features in `X`. Print the feature columns.

In [ ]:
# Your turn


<a id="task4"></a>
## Task 4: Handling the imbalanced target

Only about 14% of the women had GDM. Left alone, a model can score well by ignoring them, so we rebalance.

### Exercise 9: Show the imbalance

Print the class counts and the percentage of positive (GDM) cases in `y`, and draw a simple bar chart of the two classes.

In [ ]:
# Your turn


### Exercise 10: Oversample and undersample

Impute `X` with the median (into `X_demo`) so the samplers have no gaps, then apply `RandomOverSampler` and `RandomUnderSampler` and print the class balance after each. What is the trade-off?

> *Note:* here we resample the whole set to see the effect. In Task 5 we resample the **training data only**, after the split, which is the correct place.

In [ ]:
# Your turn


<a id="task5"></a>
## Task 5: Normalization and standardization

The final task is to put the features on a common scale. The clean way to combine imputation, scaling and resampling is a **pipeline**, fitted on the training data only, so the median fill values and the scaler's statistics are learned from the training set (no peeking at the test set) and resampling is applied only while training.

### Exercise 11: Split the data (stratified)

Split `X` (the version that still has missing values, so the pipeline can handle them) and `y` into train and test sets with `test_size=0.25`, `random_state=RANDOM_STATE`, and `stratify=y` so both sets keep the ~14% GDM rate.

In [ ]:
# Your turn


### Exercise 12: Build a leakage-safe preprocessing pipeline

Build an `imblearn` `Pipeline` with these steps: median `SimpleImputer`, `StandardScaler`, and `RandomOverSampler`. Because imbalanced-learn's pipeline can hold a resampler as a step, calling `fit_resample` on the training data runs all three in order and returns the cleaned, scaled, rebalanced training set (`X_res`, `y_res`), ready to hand to any model.

> *Hint:* use the `Pipeline` imported as `ImbPipeline`; the oversampler acts only during fitting on the training data, never on the test set.

In [ ]:
# Your turn


<a id="task6"></a>
## Task 6: Apply and evaluate classification models

The data is now model-ready. The point of all that preparation is to **predict `gestational dm`**. We train three classifiers from the classification lectures - **Logistic Regression**, a **Decision Tree** and a **Random Forest** - each inside the same leakage-safe pipeline (impute, scale, oversample the training data, then fit), and compare them.

Because the target is imbalanced (~14% GDM), we judge the models on **precision, recall, F1 and ROC AUC** for the GDM class, not on accuracy alone: a model that predicts 'no GDM' for everyone would still score about 86% accuracy while catching zero real cases.

### Exercise 13: Train the three models in leakage-safe pipelines

Build one `imblearn` `Pipeline` per model with the steps `SimpleImputer(median)`, `StandardScaler`, `RandomOverSampler`, then the classifier. Use `LogisticRegression(max_iter=1000)`, `DecisionTreeClassifier(max_depth=4)` and `RandomForestClassifier(n_estimators=200)` (all with `random_state=RANDOM_STATE`). Fit each on the **training** data.

> *Note:* the oversampler only acts during `fit` on the training data, never on the test set. Scaling makes no difference to the tree models but keeps the pipeline uniform.

In [ ]:
# Your turn


### Exercise 14: Evaluate each model on the held-out test set

For every fitted model, predict on `X_test` and print the `classification_report`, the `confusion_matrix`, and the `roc_auc_score` (using the predicted probability of the positive class). The pipeline applies the same imputation and scaling to the test set automatically.

In [ ]:
# Your turn


### Exercise 15: Compare the models and recommend one

Build a comparison table (models as rows; accuracy, precision, recall, F1 and ROC AUC as columns), sorted by F1 on the GDM class. Overlay the three ROC curves on one chart. Then, in a comment, say which model you would recommend and why, being explicit about why accuracy is the wrong headline here.

> *Note:* the dataset is very small (133 women), so the exact numbers are noisy; focus on the workflow and the reasoning, not on chasing a decimal.

In [ ]:
# Your turn


<a id="wrap"></a>
## Wrap-up

You took the raw case-study file all the way to a model comparison:

| Task | What you did |
|---|---|
| 1. Assessment | Found missing values, a near-constant column, text-coded numbers, and an imbalanced target |
| 2. Missing data | Weighed drop vs impute and chose median imputation for the 22%-missing glucose |
| 3. Enrichment | Parsed gestational age to numeric weeks, built `pulse_pressure`, and dropped ID / near-constant / leaking outcome columns |
| 4. Imbalance | Compared over- and under-sampling for the ~14% GDM target |
| 5. Normalization | Split first, then imputed, scaled and oversampled inside a pipeline fitted on the training data only |
| 6. Classification | Trained Logistic Regression, a Decision Tree and a Random Forest in leakage-safe pipelines and compared them on F1 and ROC AUC |

**The one-line lesson:** good modelling is mostly good preparation. A leakage-safe pipeline plus honest, imbalance-aware metrics (F1, recall, ROC AUC, not accuracy) is what lets you recommend a model you can defend. The exact same workflow applies to a credit or fraud dataset.

Great job. You have completed an end-to-end classification mini-project.